In [24]:
import pandas as pd

import os

os.makedirs('../data/analytics', exist_ok=True)


In [25]:
# Identify top-performing and low-performing products.



sales = pd.read_csv("../data/curated/sales_curated.csv")

product_performance = (
    sales.groupby('product_id').agg({'quantity':'sum','line_grand_total':'sum'})
    .rename(columns={'quantity':'total_units_sold','line_grand_total':'total_revenue'})
    .sort_values('total_revenue',ascending=False)
)

product_performance.head(10)

product_performance.to_csv(
    '../data/analytics/product_performance.csv',
    index=False
)


In [26]:
#products with highest sales volume.

fast_moving = (
    sales.groupby('product_id')['quantity']
    .sum()
    .sort_values(ascending=False)
)

fast_moving.head(10)

product_id
P029    47066
P010    46964
P006    46731
P011    46612
P007    46468
P021    46364
P016    46306
P028    46164
P012    45972
P019    45971
Name: quantity, dtype: int64

In [27]:
# products with little or no movement.
inventory = pd.read_csv(
    '../data/processed/inventory_master.csv'
)

slow_moving = inventory[
    inventory['current_stock']
    > inventory['reorder_level']
]

slow_moving.head()


# Dead stock:
dead_stock = (
    sales.groupby('product_id')['quantity']
    .sum()
    .reset_index()
)

dead_stock = dead_stock[
    dead_stock['quantity'] == 0
]


In [30]:
# Inventory Turnover Analysis

sales_qty = (
    sales.groupby('product_id')['quantity']
    .sum()
)

inventory_avg = (
    inventory.groupby('product_id')['current_stock']
    .mean()
)

turnover = (
    sales_qty / inventory_avg
).reset_index()

turnover.columns = [
    'product_id',
    'inventory_turnover'
]

turnover.head()

turnover.to_csv(
    '../data/analytics/inventory_turnover.csv',
    index=False
)

In [31]:
# Stock Health Assessment
inventory['stock_health'] = 'Healthy'

inventory.loc[
    inventory['current_stock']
    < inventory['safety_stock'],
    'stock_health'
] = 'Critical'

inventory.loc[
    inventory['current_stock']
    > inventory['max_stock'],
    'stock_health'
] = 'Overstock'

inventory['stock_health'].value_counts()

inventory.to_csv(
    '../data/analytics/stock_health.csv',
    index=False
)



In [32]:
#Inventory Aging Analysis
ledger = pd.read_csv(
    '../data/processed/stock_ledger.csv'
)

ledger['movement_date'] = pd.to_datetime(
    ledger['movement_date']
)

ledger['days_since_movement'] = (
    pd.Timestamp.today()
    - ledger['movement_date']
).dt.days


#Aging analysis
ledger['aging_bucket'] = pd.cut(
    ledger['days_since_movement'],
    bins=[0,30,90,180,99999],
    labels=[
        '0-30',
        '31-90',
        '91-180',
        '180+'
    ]
)

In [33]:
#Classic inventory analysis.

abc = (
    sales.groupby('product_id')
    ['line_grand_total']
    .sum()
    .sort_values(
        ascending=False
    )
    .reset_index()
)

abc['cum_pct'] = (
    abc['line_grand_total']
    .cumsum()
    /
    abc['line_grand_total'].sum()
)

abc['class'] = 'C'

abc.loc[
    abc['cum_pct'] <= 0.80,
    'class'
] = 'A'

abc.loc[
    (abc['cum_pct'] > 0.80)
    &
    (abc['cum_pct'] <= 0.95),
    'class'
] = 'B'


abc.to_csv(
    '../data/analytics/abc_classification.csv',
    index=False
)

In [34]:
#Overstock & Understock Detection

inventory['inventory_status'] = 'Normal'

inventory.loc[
    inventory['current_stock']
    < inventory['reorder_level'],
    'inventory_status'
] = 'Understock'

inventory.loc[
    inventory['current_stock']
    > inventory['max_stock'],
    'inventory_status'
] = 'Overstock'

inventory['inventory_status'].value_counts()

inventory_status
Overstock    180
Name: count, dtype: int64

In [35]:
#Product Demand Trend Analysis
sales['order_date'] = pd.to_datetime(
    sales['order_date']
)

sales['year_month'] = (
    sales['order_date']
    .dt.to_period('M')
)

demand_trend = (
    sales.groupby(
        ['year_month','product_id']
    )['quantity']
    .sum()
    .reset_index()
)

demand_trend.head()


demand_trends.to_csv(
    '../data/analytics/demand_trends.csv',
    index=False
)

NameError: name 'demand_trends' is not defined

In [ ]:
print("Analytics datasets created successfully!")

print("""
Generated:
- product_performance.csv
- inventory_turnover.csv
- abc_classification.csv
- stock_health.csv
- demand_trends.csv
""")